# 03 — Optimasi multi-objektif posisi dinding (Sec. 2 & 4 paper)

Dua objektif: `f_cost` (energi total dari surrogate, kW) dan `f_tc` (rata-rata |PMV| per penghuni, ISO 7730). Variabel keputusan = posisi dinding dalam. Logika model ada di `src/aiobep/layout.py` dan `optimise.py`. Algoritma: NSGA-II, NSGA-III, SMS-EMOA (pymoo), hiperparameter dari Tabel 3 & 6 paper.

Jalankan notebook 02 dulu (membuat `outputs/models/*.joblib`). Run penuh (5 seed × 500 generasi × 3 algoritma × 2 kasus) ±15–30 menit; set `QUICK = True` untuk uji cepat (hasil ke `outputs/_quick`, tidak menimpa hasil utama).

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "src").exists():          # notebook dijalankan dari notebooks/
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))
%matplotlib inline
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from joblib import Parallel, delayed

from aiobep.layout import OFFICE, case_study_1, case_study_2, evaluate
from aiobep.optimise import run_once
from aiobep.surrogate import Surrogate

# ---- konfigurasi ----
QUICK = False                  # True = 1 seed, 30 generasi
SEEDS = 5                      # paper: 30
GENS = 500
MODEL = "surrogate_ensemble"   # atau "surrogate_mlp"
ALGOS = ["NSGA-II", "NSGA-III", "SMS-EMOA"]
N_JOBS = -1
if QUICK:
    SEEDS, GENS = 1, 30

# (populasi, p_mutasi, eta_mutasi, p_crossover) dari Tabel 3 (kasus 1) dan Tabel 6 (kasus 2)
HP = {
    "case1_simple": {"NSGA-II": (30, 0.7, 5.0, 0.1), "NSGA-III": (40, 0.7, 5.0, 0.8), "SMS-EMOA": (40, 0.7, 5.0, 0.7)},
    "case2_watson": {"NSGA-II": (50, 0.7, 1.5, 0.7), "NSGA-III": (40, 0.7, 1.5, 0.9), "SMS-EMOA": (30, 0.8, 1.5, 0.8)},
}
out = ROOT / "outputs" / ("_quick" if QUICK else MODEL.replace("surrogate_", "opt_"))
out.mkdir(parents=True, exist_ok=True)
sur = Surrogate.load(ROOT / "outputs" / "models" / f"{MODEL}.joblib")

## Fungsi bantu

In [ ]:
def one(sc, algo, seed):
    pop, pm, eta, pc = HP[sc.name][algo]
    return sc.name, algo, seed, run_once(sc, sur, algo, seed, pop=pop, n_gen=GENS, p_m=pm, eta_m=eta, p_c=pc)


def draw_layout(ax, sc, W, area, occ, T, title):
    for j in range(sc.n_rooms):
        c = "#cfe2f3" if sc.room_types[j] == OFFICE else "#d9ead3"
        ax.add_patch(plt.Rectangle((W[j], 0), W[j + 1] - W[j], sc.B, fc=c, ec="k"))
        ax.text((W[j] + W[j + 1]) / 2, sc.B / 2, f"R{j + 1}\n{area[j]:.1f} m²\n{occ[j]} pax\n{T[j]:.1f}°C",
                ha="center", va="center", fontsize=7)
    ax.set(xlim=(0, sc.L), ylim=(0, sc.B), title=title, yticks=[])

## Jalankan kedua studi kasus
Solusi **A** = energi terendah, **B** = komfort terbaik, dipilih dari gabungan semua front (semua algoritma & seed).

In [ ]:
summary = {}
for sc in (case_study_1(), case_study_2()):
    res = Parallel(n_jobs=N_JOBS)(delayed(one)(sc, a, s) for a in ALGOS for s in range(1, SEEDS + 1))
    init = evaluate(sc, sur, sc.x_init[None, 1:-1]); f0 = init["F"][0]
    rows, fronts = [], []
    for _, algo, seed, r in res:
        F = r["F"]
        rows.append(dict(algo=algo, seed=seed, n_solutions=len(F), hypervolume_norm=r["hv"],
                         best_f_cost=F[:, 0].min() if len(F) else np.nan, best_f_tc=F[:, 1].min() if len(F) else np.nan))
        fronts += [dict(algo=algo, seed=seed, f_cost=f[0], f_tc=f[1], **{f"x{i + 1}": v for i, v in enumerate(x)})
                   for x, f in zip(r["X"], F)]
    runs, front = pd.DataFrame(rows), pd.DataFrame(fronts)
    runs.to_csv(out / f"{sc.name}_runs.csv", index=False); front.to_csv(out / f"{sc.name}_fronts.csv", index=False)
    agg = runs.groupby("algo").agg(hv_mean=("hypervolume_norm", "mean"), hv_std=("hypervolume_norm", "std"),
                                   n_sol=("n_solutions", "mean"), f_cost=("best_f_cost", "mean"), f_tc=("best_f_tc", "mean"))
    print(f"\n== {sc.name}: awal f_cost={f0[0]:.3f} kW, f_tc={f0[1]:.3f}"); display(agg.round(4))

    xcols = [c for c in front.columns if c.startswith("x")]
    sols = {"A (min energy)": front.loc[front.f_cost.idxmin()], "B (best comfort)": front.loc[front.f_tc.idxmin()]}
    info = dict(initial=dict(f_cost=float(f0[0]), f_tc=float(f0[1])), aggregate=agg.round(5).to_dict())
    fig, axs = plt.subplots(3, 1, figsize=(10, 7.5))
    draw_layout(axs[0], sc, init["W"][0], init["area"][0], init["occ"][0], init["T"][0],
                f"Awal: f_cost={f0[0]:.3f} kW, f_tc={f0[1]:.3f}")
    for ax, (k, s) in zip(axs[1:], sols.items()):
        r = evaluate(sc, sur, s[xcols].to_numpy(float)[None]); fc, ft = r["F"][0]
        info[k] = dict(f_cost=float(fc), f_tc=float(ft), energy_saving_pct=float((1 - fc / f0[0]) * 100),
                       comfort_gain_pct=float((1 - ft / f0[1]) * 100), algo=s["algo"], seed=int(s["seed"]),
                       walls=[round(float(w), 2) for w in r["W"][0]], areas=r["area"][0].round(1).tolist(),
                       occupants=r["occ"][0].tolist())
        draw_layout(ax, sc, r["W"][0], r["area"][0], r["occ"][0], r["T"][0],
                    f"Solusi {k}: f_cost={fc:.3f} ({info[k]['energy_saving_pct']:+.2f}% hemat), "
                    f"f_tc={ft:.3f} ({info[k]['comfort_gain_pct']:+.2f}% lebih baik)")
    plt.tight_layout(); plt.savefig(out / f"{sc.name}_layouts.png", dpi=130); plt.show()
    json.dump(info, open(out / f"{sc.name}_summary.json", "w"), indent=2)
    summary[sc.name] = info

    fig, ax = plt.subplots(figsize=(6, 4.5))
    for algo, g in front.groupby("algo"):
        ax.scatter(g.f_cost, g.f_tc, s=10, alpha=.5, label=algo)
    ax.scatter(*f0, c="k", marker="*", s=150, label="layout awal")
    ax.set(xlabel="f_cost [kW, surrogate]", ylabel="f_tc [rata-rata |PMV| per penghuni]",
           title=f"{sc.name}: front non-dominated ({SEEDS} seed)")
    ax.legend(); plt.tight_layout(); plt.savefig(out / f"{sc.name}_pareto.png", dpi=130); plt.show()

## Ringkasan hemat energi / perbaikan komfort

In [ ]:
pd.DataFrame({c: {k: (round(v["energy_saving_pct"], 2), round(v["comfort_gain_pct"], 2))
                  for k, v in s.items() if k[0] in "AB"} for c, s in summary.items()}).rename_axis("(hemat energi %, komfort %)")